In [27]:
import pandas as pd
import numpy as np

In [90]:
df = pd.read_csv('ATLAS_GDC-CHEOPS.dat', sep= '\s+', skiprows=1)
df1 = df.iloc[::2]
df1.columns = ['Z', 'Vel', 'logg', 'logTeff', 'y1']
df2 = df.iloc[1::2]
df2.columns = ['Z', 'Vel', 'logg', 'logTeff', 'y2']

df  = pd.merge(df1, df2, on=['Z', 'Vel', 'logg', 'logTeff'])
df['y'] = 0.3*df['y1'] + df['y2'] 

df= df[['Z', 'logg', 'logTeff', 'y']]
df

,Z,logg,logTeff,y
0,0.0,0.0,3.544,0.76919
1,0.0,0.0,3.574,0.62194
2,0.0,0.0,3.602,0.50962
3,0.0,0.0,3.628,0.45970
4,0.0,0.0,3.653,0.42691
...,...,...,...,...
9570,0.0,5.0,4.663,0.08088
9571,0.0,5.0,4.672,0.08076
9572,0.0,5.0,4.681,0.08034
9573,0.0,5.0,4.690,0.07995


In [1]:
from Planet_tools.phasecurve import gravity_darkening_coefficient, gravity_darkening_coefficient_old
import numpy as np
from uncertainties.umath import log10

In [8]:
gravity_darkening_coefficient(logg=(0, 0.01), 
								Teff=(10**3.544,0.001), 
                                band="CHEOPS", beta1_ch=0.2)

0.5050600000000001+/-0.0003152597756014286

In [11]:
gravity_darkening_coefficient(logg=(3, 0.01), 
								Teff=(10**3.5441,0.001), 
                                band="JWST/PRISM")

0.13309+/-0.00012314464789003887

In [92]:
# get value of y for given Z, logg, logTeff no interpolation
def get_y(logTeff, logg, Z=None):
	if Z:
		row = df[ (df['logg'] == logg) & (df['logTeff'] == logTeff) & (df['Z'] == Z)]
	else:
		row = df[ (df['logTeff'] == logTeff) & (df['logg'] == logg)]                    
	if not row.empty:
		return row['y'].values[0]
	else:
		return np.nan	

get_y(3.544, 0.0, 0)

0.76919

In [93]:
z_list 		 = sorted(df['Z'].unique())
logg_list 	 = sorted(df['logg'].unique())
logTeff_list = sorted(df['logTeff'].unique())

dff     = np.empty((len(logTeff_list),len(logg_list),len(z_list)))
Z = True

for i,log_T in enumerate(logTeff_list):
	for j,log_g in enumerate(logg_list):
		if Z:
			for k,_z in enumerate(z_list):
				y_val = get_y(log_T, log_g, _z)
				dff[i,j,k] = y_val
		else:
			y_val = get_y(log_T, log_g)
			dff[i,j] = y_val
			# print(f"Z: {Z}, logg: {logg}, logTeff: {logTeff} => y: {y_val}")

dff.shape

(79, 11, 19)

In [95]:
# 3d spline interpolation to get y for given Z, logg, logTeff
from scipy.interpolate import RegularGridInterpolator
interp_func = RegularGridInterpolator(  points = (logTeff_list, logg_list, z_list) if Z else (logTeff_list, logg_list), 
										values = dff,
										bounds_error=False, method='linear')

interp_func((3.544, 0.0, 0)).item()

0.76919

In [96]:
np.nanmedian(norm(4,5).rvs(1000))

4.022328331816066

In [77]:
# 3d interpolation to get y for given Z, logg, logTeff
from scipy.interpolate import LinearNDInterpolator

# The data does not form a complete regular grid (9348 points vs 16511 required), 
# so we use LinearNDInterpolator instead of RegularGridInterpolator.

# Group by coordinates to handle any duplicates (e.g. varying Vel) and ensure unique points
df_gb = df.groupby(['Z', 'logg', 'logTeff'])['y'].mean().reset_index()

interp_func = LinearNDInterpolator(df_gb[['Z', 'logg', 'logTeff']].values, df_gb['y'].values,rescale=True)

interp_func(0.0, 0, 3.544)

array(0.761984)

In [37]:
df.groupby(['Z', 'logg', 'logTeff'])['y'].mean()

Z     logg  logTeff
-5.0  0.0   3.544      0.72552
            3.574      0.64788
            3.602      0.57748
            3.628      0.51199
            3.653      0.45418
                        ...   
 1.0  5.0   4.505      0.10723
            4.519      0.10329
            4.531      0.09887
            4.544      0.08899
            4.574      0.09282
Name: y, Length: 7698, dtype: float64

In [ ]:
get value of y for given Z, logg, logTeff
    

In [33]:
df.pivot_table(index='Z', columns=['Z','logg', 'logTeff'], values='y')

Z           -5.0                                                       \
logg         0.0                                                        
logTeff    3.544    3.574    3.602    3.628    3.653    3.677   3.699   
Z                                                                       
-5.0     0.72552  0.64788  0.57748  0.51199  0.45418  0.40787  0.3719   
-4.5         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-4.0         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-3.5         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-3.0         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-2.5         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-2.0         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-1.5         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-1.0         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-0.5         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-0.3         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-0.2         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
-0.1         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
 0.0         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
 0.1         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
 0.2         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
 0.3         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
 0.5         NaN      NaN      NaN      NaN      NaN      NaN     NaN   
 1.0         NaN      NaN      NaN      NaN      NaN      NaN     NaN   

Z                                   ...      1.0                           \
logg                                ...      5.0                            
logTeff    3.720    3.740    3.760  ...    4.431  4.447    4.462    4.477   
Z                                   ...                                     
-5.0     0.34619  0.36515  0.30448  ...      NaN    NaN      NaN      NaN   
-4.5         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-4.0         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-3.5         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-3.0         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-2.5         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-2.0         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-1.5         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-1.0         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-0.5         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-0.3         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-0.2         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
-0.1         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
 0.0         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
 0.1         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
 0.2         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
 0.3         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
 0.5         NaN      NaN      NaN  ...      NaN    NaN      NaN      NaN   
 1.0         NaN      NaN      NaN  ...  0.11528  0.116  0.11608  0.11477   

Z                                                              
logg                                                           
logTeff    4.491    4.505    4.519    4.531    4.544    4.574  
Z                                                              
-5.0         NaN      NaN      NaN      NaN      NaN      NaN  
-4.5         NaN      NaN      NaN      NaN      NaN      NaN  
-4.0         NaN      NaN      NaN      NaN      NaN      NaN  
-3.5         NaN      NaN      NaN      NaN      NaN      NaN  
-3.0         NaN      NaN      NaN  

In [22]:
df.pivot_table(index='Z', columns=['logg', 'logTeff'], values='y').values.reshape(len(df['Z'].unique()), len(df['logg'].unique()), len(df['logTeff'].unique())),


ValueError: cannot reshape array of size 9348 into shape (19,11,79)